# 03 · Retrieval and consistency checking

Steps 3 and 4 on a tiny two-chapter story, end to end: extract each chapter, store it in the Neo4j story bible, pull the history of every entity the new chapter mentions, and ask the checker whether any new fact contradicts it. Each flag comes with a cited reason and a confidence tier (high / medium / low).

**Before running:** Ollama running, `.env` filled in, kernel **Continuity Editor (.venv)**.

## 0. Setup

In [ ]:
import importlib.util
import sys

print(f"Python {sys.version.split()[0]} at {sys.executable}")
if sys.version_info < (3, 11):
    raise RuntimeError("This kernel's Python is too old: click the kernel name (top right) and choose 'Continuity Editor (.venv)'.")

# Libraries this notebook uses; continuity_editor is this project's own code.
REQUIRED = ["pandas", "ollama", "rapidfuzz", "neo4j", "pydantic", "pydantic_settings", "continuity_editor"]
missing = [name for name in REQUIRED if importlib.util.find_spec(name) is None]
if missing:
    print("Installing missing libraries:", ", ".join(missing))
    %pip install --quiet -r ../requirements.txt -e ..
    print("Done. Restart the kernel (Restart in the toolbar), then run the notebook again.")
else:
    print("All libraries are installed.")

## 1. A two-chapter story with one mistake

Chapter 1 gives Mira blue eyes; chapter 2 gives her green eyes, with nothing in the story to explain it.

In [ ]:
import pandas as pd

from continuity_editor.extraction import extract_novel
from continuity_editor.extraction.chapters import ChapterText
from continuity_editor.graph import Neo4jConnection, apply_schema, reset_novel
from continuity_editor.llm import LLMClient
from continuity_editor.paths import extraction_cache_dir
from continuity_editor.pipeline import run_novel
from continuity_editor.retrieval import fact_history

pd.set_option("display.max_colwidth", 140)
DEMO = "demo"
chapters = [
    ChapterText(1, "The Kestrel",
        "Captain Mira Voss stood at the bow of the Kestrel. Her blue eyes were sharp in the lamplight, and at "
        "twenty-five she was the youngest captain in the fleet. Her first mate, old Tobias, had sailed with her "
        "father for thirty years."),
    ChapterText(2, "Port Linden",
        "Three days later the Kestrel reached the harbour at Port Linden. Mira's green eyes narrowed at the "
        "harbourmaster, who demanded twice the usual fee. Tobias, grumbling, counted out the silver."),
]

## 2. Extract, store and check (a minute or two the first time)

In [ ]:
llm = LLMClient()
llm.verify()
extractions = extract_novel(llm, DEMO, chapters, extraction_cache_dir())

conn = Neo4jConnection()
apply_schema(conn)
flags = run_novel(conn, llm, DEMO, "Demo story", chapters, extractions)
pd.DataFrame([
    {"chapter": f.chapter_number, "entity": f.entity_name, "new": f"{f.new_attribute}: {f.new_value}",
     "earlier": f"ch{f.prior_chapter} {f.prior_attribute}: {f.prior_value}", "tier": f.confidence,
     "reason": f.reason}
    for f in flags
])

## 3. Targeted retrieval

What the checker saw for each entity chapter 2 mentions: only the facts established before chapter 2, each with its chapter and quote.

In [ ]:
entity_ids = [row["id"] for row in conn.query(
    "MATCH (:Chapter {chapter_id: $ch})-[:MENTIONS]->(e:Entity) RETURN e.entity_id AS id", ch=f"{DEMO}:ch2")]
history = fact_history(conn, entity_ids, before_chapter=2)
pd.DataFrame([p.model_dump(exclude={"fact_id"}) for facts in history.values() for p in facts])

## 4. The flag in the graph

The contradiction is stored as a `CONTRADICTS` relationship between the two facts, so the story bible keeps a record of it. Then the demo story is deleted.

In [ ]:
display(pd.DataFrame(conn.query(
    "MATCH (e:Entity)-[:HAS_FACT]->(new:Fact)-[c:CONTRADICTS]->(old:Fact) WHERE new.novel_id = $n "
    "RETURN e.name AS entity, new.value AS new_value, old.value AS earlier_value, c.confidence AS tier", n=DEMO)))
reset_novel(conn, DEMO)
conn.close()

## 5. Flags on the real book

After `python scripts/evaluate.py` has run, its flags for the test corpus (the book with injected contradictions) are saved in `data/results/`.

In [ ]:
from continuity_editor.paths import results_dir
from continuity_editor.pipeline import load_flags

path = results_dir() / "flags_oz_injected.json"
if path.exists():
    book_flags = load_flags(path)
    display(pd.DataFrame([
        {"chapter": f.chapter_number, "entity": f.entity_name, "new": f"{f.new_attribute}: {f.new_value}",
         "earlier": f"ch{f.prior_chapter} {f.prior_attribute}: {f.prior_value}", "tier": f.confidence,
         "reason": f.reason}
        for f in book_flags
    ]))
else:
    print("No results yet: run `python scripts/evaluate.py` first.")